# TurboSWAN: Track and Compare Your Experiments

A model is rarely trained once. You adjust the learning rate, add data, or train longer, and weeks later you need to know which run did what and how well it worked. This notebook trains a small TurboSWAN model and records it in MLflow: a lab notebook for training runs that fills itself in.

**What this gives you:**

- **Watch training as it happens.** Training and validation loss appear after every epoch, so you can see whether a run is learning without waiting for it to finish.
- **Compare runs side by side.** Overlay learning curves and see exactly which settings differed.
- **See where a model errs.** Each run saves an error map for a validation example.
- **Come back later.** Every run keeps its recipe, settings, data references, and fitted model.
- **Mark models for follow-up.** Tag runs as `promising` or `needs-expert-review`, then filter on those tags later.

Continuing training from a saved model is planned.

## Before You Start

**Run locally:** no Data & AI platform account is needed. Follow the [getting started guide](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-getting-started/) first: it shows how to get this code (branch `integration/turboswan-mlflow-sweep`), install it, choose the notebook kernel, and copy the demo data to `~/turboswan/data/demo` (access requires the Deltares-research organization). Then make sure the notebook's working directory is `projects/turboswan/notebooks` and run the cells in order.

**Or run on the Data & AI platform:** with platform access, you can run this notebook in JupyterHub and record the experiment in platform MLflow. Follow [Prepare JupyterHub](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-platform-setup/), then [Run the tracked demo](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-mlflow-platform/) for the kernel, data and recipe changes (`run.mode = "dnai"`). These new documentation pages are pending publication; the links may not work yet. This getting-started route runs training on the JupyterHub server, not on the planned Snellius HPC environment.

Every choice in this notebook comes from a **recipe**, a small text file (TOML). Keep the [TOML configuration guide](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/) open next to the notebook: it explains each setting (access requires the Deltares-research organization).

The small model and short training schedule demonstrate the workflow; they are not a recommendation for a scientific experiment.

In [ ]:
import sys
from pathlib import Path
from turboswan import training
from turboswan.training_mlflow import track_training_run

## 1. Choose a Recipe

`demo_mlflow_local.toml` holds every choice for this run: the data, the model, how it learns, and how the run is recorded. Copy it before changing anything, and point `RECIPE_FILE` below to your copy. Run the next cell, then read the parts of the recipe printed below it, exactly as written in the file.

In [ ]:
RECIPE_FILE = Path("../configs/demo_mlflow_local.toml")

recipe = training.load_config(RECIPE_FILE)

### The data

`source` names an existing data folder on the machine running training: your own machine or your JupyterHub server. Both `local` and `dnai` modes read its `train_data` and `val_data` subfolders; `run.mode` selects the MLflow destination, not the data location. The variable names must match the TFRecords and the normalization file.

In [ ]:
training.show_recipe_section(recipe, "data")

### The model and how it learns

The demo model is deliberately small and trains for only a few epochs, so it finishes in minutes. Its numbers are for demonstration, not a recommendation.

In [ ]:
training.show_recipe_section(recipe, "model")
print()
training.show_recipe_section(recipe, "training")

### How the run is recorded

`[run]` says where this run's files go. `[mlflow]` names the study (`experiment_name`, which groups related runs), the saved model (`model_name`), and gives this run a short description. The demo also plots one validation example as an error map; this is not an independent scientific evaluation. The comparison names a folder inside the data source (`subdir`), so it follows the data location.

In [ ]:
training.show_recipe_section(recipe, "run")
print()
training.show_recipe_section(recipe, "mlflow")
print()
training.show_recipe_section(recipe, "comparisons")

## 2. Optional: Watch Training Live

The notebook is busy while it trains, so open MLflow first. Run the next cell. It starts MLflow in the background; after a few seconds, open http://127.0.0.1:5001 in your browser. MLflow keeps running until you close this notebook's kernel or VS Code, so you only need to run the cell once.

After training starts, open your study and the active run. Its **Model metrics** charts gain a point after each epoch; refresh the page to update them. This demo has only three short epochs, but the same view lets you check a long run early and decide whether it is worth continuing. The error map and saved model appear when training finishes.

In [ ]:
import subprocess

if recipe.run.mode == "local":
    record_folder = (recipe.mlflow.local_tracking_dir or Path.home() / "turboswan" / "mlflow").expanduser().resolve()
    record_folder.mkdir(parents=True, exist_ok=True)
    log_path = record_folder / "mlflow-ui.log"
    with log_path.open("a") as log_file:
        subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--port", "5001", "--backend-store-uri", f"sqlite:///{record_folder / 'mlflow.db'}"], stdout=log_file, stderr=subprocess.STDOUT)
    print("MLflow is starting. In a few seconds, open http://127.0.0.1:5001 in your browser.")
    print(f"MLflow server logs: {log_path}")
else:
    print("This recipe records to the Data & AI platform's MLflow. Open it there.")

## 3. Train and Record

The next cell trains the configured model. Losses are sent to MLflow after each epoch. When training finishes, the learning curve, error map, settings, and fitted model are saved with the run. It may take a few minutes.

In [ ]:
result = track_training_run(recipe)

print(f"Completed run: {result.run_id}")
print(f"Saved model: {result.model_uri}")

## 4. What You Now Have

Open MLflow (started in step 2; rerun that cell if it is no longer running), then select your run under the recipe's `experiment_name`. The record answers the questions you will ask later:

- **What did I run?** *Parameters* list the selected data, model, and training choices. Under *Artifacts*, `metadata/recipe.toml` is the recipe exactly as written and `metadata/config.json` holds every effective setting, including defaults.
- **How did it learn?** *Model metrics* show training and validation masked MSE in normalised wave height after each epoch; lower is better. A widening gap between them can indicate overfitting.
- **Where does it err?** `comparisons/01/error-map-hs.png` maps the prediction error for one validation example.
- **Can I use the model again?** The fitted Keras model is saved with its expected input and output shapes.

The data manifest records the selected shard names, not checksums of their contents. These records make runs comparable; they do not establish scientific validity.

In [ ]:
print("To browse your runs, run this in a terminal:")
print(f'"{sys.executable}" -m mlflow ui --backend-store-uri "{result.tracking_uri}"')

## 5. Compare Experiments

One run is a record; several runs make a comparison. Now that you have looked at your first run in MLflow, come back here and change the recipe:

1. Open `demo_mlflow_local.toml` and, under `[training]`, change `n_epochs` to e.g. `5`. Try a different `learning_rate` too.
2. Under `[mlflow]`, change `description` to say what you changed, and keep the same `experiment_name` so the runs stay together. Save the file.
3. Rerun this notebook from the cell that loads the recipe (`training.load_config`) down to the cell that trains. The recipe is read again when it loads, so you do not need to restart the notebook. The cells that print the recipe show your new values.
4. Refresh MLflow. Your new run appears next to the first one, which keeps its own copy of the old recipe under `metadata/recipe.toml`.

The MLflow cell does not need to run again. Then, in MLflow, open your study and:

- select two or more runs and choose **Compare** to overlay their learning curves and see which settings differ;
- switch the run list to its chart view to see a metric for every run at once;
- open `comparisons/01/error-map-hs.png` in each run to compare errors on the same validation example.

A single example shows where models differ, not which one is better overall.

## 6. Mark Runs for Follow-Up

Tags are short labels you add after looking at a result, so you can find it again among many runs. Do this in MLflow: open the run, find **Tags** in its overview, and add one, for example `research_status` = `promising`. To find tagged runs later, type `tags.research_status = 'promising'` in the run search box.

Use your own vocabulary, consistently within a study. Tags are your bookkeeping, not a scientific validation.

## Come Back Later

Your records stay in `~/turboswan/mlflow` after you close this notebook. To revisit them, run the MLflow cell in step 2 again; nothing needs retraining. To repeat an old experiment, download its `metadata/recipe.toml`, save it as a new recipe, and select it with `RECIPE_FILE`.

Continuing training from a saved model is planned; for now, each run trains from scratch.